# UK Political Donations Analysis (2010–present)
**Source:** Donation Watch  
**Goal:** Understand where parties get their money, what donor types they rely on, and flag high-value donors for further investigation.

---

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from pathlib import Path

# ── Style ──────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 150,
    'figure.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

OUTPUT_DIR = Path('article/figures')
OUTPUT_DIR.mkdir(exist_ok=True)

PARTY_COLOURS = {
    'Conservative': '#0087DC',
    'Labour':       '#E4003B',
    'Liberal Democrats': '#FAA61A',
    'SNP':          '#FDF38E',
    'Reform UK':    '#12B6CF',
    'Green Party':  '#02A95B',
    'Other':        '#AAAAAA',
}

## 1 · Load & Clean

In [ ]:
# ── Load ───────────────────────────────────────────────────────────────────
CSV_PATH = 'data/DonationWatch-donations-unitedkingdom-2026-06-09.csv'   # ← change to your file path

df = pd.read_csv(CSV_PATH, parse_dates=['Date'])
df.columns = df.columns.str.strip()

# Normalise column names defensively
df.rename(columns={
    'Donor Name': 'donor_name',
    'Receiver':   'party',
    'Amount':     'amount',
    'Currency':   'currency',
    'Donor Type': 'donor_type',
    'Date':       'date',
}, inplace=True)

# GBP only (drop foreign currency donations or convert separately)
df = df[df['currency'] == 'GBP'].copy()
df['year'] = df['date'].dt.year

# Consolidate minor parties into 'Other'
major_parties = list(PARTY_COLOURS.keys())[:-1]   # all except 'Other'
df['party_group'] = df['party'].where(df['party'].isin(major_parties), other='Other')

print(f"Rows: {len(df):,}  |  Date range: {df['date'].min().date()} → {df['date'].max().date()}")
df.head(3)

## 2 · Total donations per party

In [ ]:
party_totals = (
    df.groupby('party_group')['amount']
    .sum()
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(9, 5))
colours = [PARTY_COLOURS.get(p, '#AAAAAA') for p in party_totals.index]
bars = ax.barh(party_totals.index, party_totals.values / 1e6, color=colours, edgecolor='white')
ax.set_xlabel('Total donations (£ millions)')
ax.set_title('Total UK political donations by party (2010–present)')
ax.bar_label(bars, fmt='£%.1fM', padding=4, fontsize=9)
ax.set_xlim(0, party_totals.values.max() / 1e6 * 1.18)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '01_total_by_party.png')
plt.show()

## 3 · Donor type mix per party (stacked bar)

In [ ]:
pivot = (
    df.groupby(['party_group', 'donor_type'])['amount']
    .sum()
    .unstack(fill_value=0)
)

# Normalise to percentages
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
pivot_pct = pivot_pct.loc[party_totals.index[::-1]]  # same order as chart 1

DONOR_COLOURS = {
    'Individual':   '#4C72B0',
    'Company':      '#DD8452',
    'Trade Union':  '#55A868',
    'Unincorporated Association': '#C44E52',
    'Public Fund':  '#8172B2',
    'Other':        '#CCB974',
}

fig, ax = plt.subplots(figsize=(10, 5))
left = np.zeros(len(pivot_pct))
for donor_type in pivot_pct.columns:
    colour = DONOR_COLOURS.get(donor_type, '#999999')
    ax.barh(pivot_pct.index, pivot_pct[donor_type], left=left,
            color=colour, label=donor_type, edgecolor='white')
    left += pivot_pct[donor_type].values

ax.set_xlabel('Share of donations (%)')
ax.set_title('Donor type mix per party')
ax.legend(loc='lower right', fontsize=8, framealpha=0.5)
ax.set_xlim(0, 100)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '02_donor_type_mix.png')
plt.show()

## 4 · Donations over time (annual, by party)

In [ ]:
annual = (
    df.groupby(['year', 'party_group'])['amount']
    .sum()
    .unstack(fill_value=0)
    / 1e6
)

fig, ax = plt.subplots(figsize=(12, 5))
for party in annual.columns:
    ax.plot(annual.index, annual[party],
            label=party, color=PARTY_COLOURS.get(party, '#AAAAAA'),
            linewidth=2, marker='o', markersize=4)

ax.set_ylabel('£ millions')
ax.set_title('Annual donations to UK parties over time')
ax.legend(fontsize=8, framealpha=0.5)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:.0f}M'))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '03_annual_trend.png')
plt.show()

## 5 · Top 20 donors overall (flag for investigation)

In [ ]:
top_donors = (
    df.groupby(['donor_name', 'donor_type'])['amount']
    .sum()
    .reset_index()
    .sort_values('amount', ascending=False)
    .head(20)
    .reset_index(drop=True)
)
top_donors['amount_fmt'] = top_donors['amount'].map('£{:,.0f}'.format)

fig, ax = plt.subplots(figsize=(10, 7))
ax.axis('off')
table = ax.table(
    cellText=top_donors[['donor_name', 'donor_type', 'amount_fmt']].values,
    colLabels=['Donor', 'Type', 'Total donated'],
    cellLoc='left', loc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.auto_set_column_width([0, 1, 2])
ax.set_title('Top 20 donors (all parties combined)', pad=12, fontsize=12)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '04_top_donors_table.png')
plt.show()

# Also export as CSV for follow-up research
top_donors.drop(columns='amount_fmt').to_csv(OUTPUT_DIR / 'top_donors.csv', index=False)
print('Saved top_donors.csv')

## 6 · High-value donor flag list (for Phase 2 investigation)

Export all donors above a threshold for cross-referencing with Companies House, Land Registry, etc.

In [ ]:
THRESHOLD = 50_000  # £50k total — adjust as needed

flagged = (
    df.groupby(['donor_name', 'donor_type'])
    .agg(
        total_donated=('amount', 'sum'),
        num_donations=('amount', 'count'),
        parties_donated_to=('party', lambda x: ', '.join(sorted(x.unique()))),
        first_donation=('date', 'min'),
        last_donation=('date', 'max'),
    )
    .reset_index()
    .query('total_donated >= @THRESHOLD')
    .sort_values('total_donated', ascending=False)
)

flagged.to_csv(OUTPUT_DIR / 'flagged_donors.csv', index=False)
print(f"{len(flagged):,} donors above £{THRESHOLD:,} threshold → saved flagged_donors.csv")
flagged.head(10)

---
## Next steps (Phase 2)

With `flagged_donors.csv` in hand:

1. **Companies** → query [Companies House API](https://developer.company-information.service.gov.uk/) for financials, directors, PSC (beneficial ownership)
2. **Individuals** → cross-reference with [Land Registry HMLR](https://use-land-property-data.service.gov.uk/) and [OCCRP Aleph](https://aleph.occrp.org/) (Offshore Leaks)
3. **Both** → check [OpenSanctions](https://www.opensanctions.org/) for PEP/sanctions hits

A Phase 2 notebook can automate the Companies House lookups in bulk via their free REST API.